<img src="https://devra.ai/analyst/notebook/4517/image.jpg" style="width: 100%; height: auto;" />

<div style="text-align:center; border-radius:15px; padding:15px; color:white; margin:0; font-family: 'Orbitron', sans-serif; background: #2E0249; background: #11001C; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.3); overflow:hidden; margin-bottom: 1em;">
  <div style="font-size:150%; color:#FEE100"><b>Global Olympic Athletes Performance Analysis</b></div>
  <div>This notebook was created with the help of <a href="https://devra.ai/ref/kaggle" style="color:#6666FF">Devra AI</a></div>
</div>

# Table of Contents

- [Introduction](#Introduction)
- [Data Cleaning and Preprocessing](#Data-Cleaning-and-Preprocessing)
- [Exploratory Data Analysis](#Exploratory-Data-Analysis)
- [Exploratory Visualizations](#Exploratory-Visualizations)
- [Correlation Analysis](#Correlation-Analysis)
- [Prediction Model](#Prediction-Model)
- [Conclusion](#Conclusion)

# Introduction

It is fascinating how a single dataset capturing global Olympic athletes can reveal so many insights regarding performance trends, demographics, and even the dynamics of sports over time. In this notebook, we explore the dataset to unearth hidden relationships. If you find this analysis useful, please consider upvoting it.

In [ ]:
# Import necessary libraries and suppress warnings
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Ensure matplotlib uses the 'Agg' backend
import matplotlib.pyplot as plt
plt.switch_backend('Agg')  # If only using plt, enforce Agg backend
%matplotlib inline

import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.inspection import permutation_importance

# Set seaborn style
sns.set(style='whitegrid')


In [ ]:
# Load the dataset
data_path = '/kaggle/input/global-olympic-athletes-performance-dataset/dataset_olympics.csv'
df = pd.read_csv(data_path, encoding='ascii', delimiter=',')

# Display first few rows of the dataset to verify loading
print('Dataset loaded successfully. Here are the first few rows:')
df.head()

In [ ]:
# Data Cleaning and Preprocessing
# We create a binary column 'Medal_Awarded' to indicate whether an athlete won a medal
df['Medal_Awarded'] = np.where(df['Medal'].isnull() | (df['Medal'].str.strip()=='') | (df['Medal'].str.upper()=='NA'), 0, 1)

# Check columns and basic info
print('DataFrame info:')
df.info()

# Checking missing values per column
missing_vals = df.isnull().sum()
print('Missing values by column:')
print(missing_vals)

# Converting Year column to datetime type by inferring a date
# Since Year is an integer, we convert it to datetime assuming January 1st of that year
df['Year_dt'] = pd.to_datetime(df['Year'].astype(str) + '-01-01')

# A note for future notebook developers: Always check and convert date columns when they are provided as integers
print('Conversion to datetime completed.')

In [ ]:
# Exploratory Data Analysis

# Summary statistics for numerical columns
print('Summary statistics:')
display(df.describe())

# Distribution of Medal Awarded
print('Medal Awarded distribution:')
display(df['Medal_Awarded'].value_counts())

# Grouping by Sex for a quick check on demographic differences
print('Group by Sex:')
display(df.groupby('Sex')['Medal_Awarded'].mean())

In [ ]:
# Exploratory Visualizations

# 1. Age Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['Age'].dropna(), kde=True, bins=30, color='skyblue')
plt.title('Age Distribution of Athletes')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# 2. Weight Distribution via Box and Violin Plots
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.boxplot(x=df['Weight'].dropna(), color='lightgreen')
plt.title('Box Plot of Athlete Weights')

plt.subplot(1, 2, 2)
sns.violinplot(x=df['Weight'].dropna(), color='lightpink')
plt.title('Violin Plot of Athlete Weights')

plt.tight_layout()
plt.show()

# 3. Medal Awarded Count (Pie Chart approximation using countplot)
plt.figure(figsize=(7, 5))
sns.countplot(x=df['Medal_Awarded'], palette='pastel')
plt.title('Count of Medal Awarded (0 = No Medal, 1 = Medal Awarded)')
plt.xlabel('Medal Awarded')
plt.ylabel('Number of Athletes')
plt.show()

# 4. Grouped Barplot: Medal Awarded by Season
season_medal = df.groupby('Season')['Medal_Awarded'].mean().reset_index()
plt.figure(figsize=(8, 5))
sns.barplot(x='Season', y='Medal_Awarded', data=season_medal, palette='muted')
plt.title('Average Medal Award Rate by Season')
plt.ylabel('Average Medal Award Rate')
plt.show()

# 5. Pair Plot for numerical features (Age, Height, Weight)
pair_data = df[['Age', 'Height', 'Weight']].dropna()
sns.pairplot(pair_data, diag_kind='kde')
plt.suptitle('Pair Plot of Age, Height, and Weight', y=1.02)
plt.show()

In [ ]:
# Correlation Analysis

# Select only numeric columns for the correlation heatmap
numeric_df = df.select_dtypes(include=[np.number])

# Proceed with a correlation heatmap only if we have four or more numeric columns
if numeric_df.shape[1] >= 4:
    corr = numeric_df.corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
    plt.title('Correlation Heatmap of Numeric Features')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough numeric columns for a correlation heatmap.')

In [ ]:
# Prediction Model
# Here we build a simple logistic regression model to predict whether an athlete wins a medal.

# For simplicity, we use numerical features and encoded categorical features
model_df = df[['Age', 'Height', 'Weight', 'Sex', 'Medal_Awarded']].dropna()

# Encode 'Sex' column: assuming values like 'M' and 'F'
le = LabelEncoder()
model_df['Sex_encoded'] = le.fit_transform(model_df['Sex'])

# Define features and target
X = model_df[['Age', 'Height', 'Weight', 'Sex_encoded']]
y = model_df['Medal_Awarded']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Build logistic regression model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Calculate prediction accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Logistic Regression Accuracy: {accuracy:.2f}')

In [ ]:
# Evaluate Prediction Model

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# 2. ROC Curve
y_prob = model.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# 3. Permutation Importance
perm_importance = permutation_importance(model, X_test_scaled, y_test, n_repeats=10, random_state=42)
feature_names = ['Age', 'Height', 'Weight', 'Sex_encoded']
sorted_idx = perm_importance.importances_mean.argsort()
plt.figure(figsize=(8, 5))
plt.barh(np.array(feature_names)[sorted_idx], perm_importance.importances_mean[sorted_idx], color='teal')
plt.xlabel('Permutation Importance')
plt.title('Feature Importance based on Permutation')
plt.tight_layout()
plt.show()

# Conclusion

This notebook presented a thorough exploration of the global Olympic athletes performance dataset. We started by cleaning and preprocessing the data, followed by multiple visualizations that revealed interesting trends in athlete demographics, performance, and medal outcomes. A simple logistic regression model was built to predict medal awards; while the accuracy was modest, it offers a baseline upon which more advanced models (e.g., ensemble techniques or deep learning) could be explored.

Future analysis could include:
- Incorporating additional features such as sports or events through advanced encoding techniques
- Analyzing temporal trends by splitting the data into different eras
- Experimenting with more sophisticated models and hyperparameter tuning

If you enjoyed this notebook and found it useful, please consider upvoting it.